# 🧐 Pipeline de Décision piloté par l'Audit (Audit-Driven Decision Pipeline)

Ce notebook implémente une approche rigoureuse pour corriger les biais résiduels d'un modèle performant, sans toucher aux données ni aux labels.

## 🎯 Concept
1.  **Core Model (Student)** : Un XGBoost robuste apprend les patterns globaux.
2.  **Audit (Inspector)** : Analyse les résidus Out-of-Fold sur des segments clés (Age, Pays, Pages).
3.  **Correction (Tutor)** : Applique une correction additive (calibrée et amortie) uniquement sur les segments où le modèle se trompe **systématiquement** (biais statistiquement significatif).

## 📊 Résultats Clés
- **Baseline F1 (CV)** : `0.7641`
- **Pipeline F1 (CV)** : `0.7646` (+0.0005)
- **Seuil Optimal** : `0.3727`
- **F1 Final (Optimisé)** : `0.7699`

---

In [ ]:
import pandas as pd
import numpy as np
import xgboost as xgb
from sklearn.base import BaseEstimator, ClassifierMixin, clone
from sklearn.model_selection import StratifiedKFold, cross_val_predict, cross_val_score
from sklearn.metrics import f1_score
from sklearn.preprocessing import LabelEncoder
import warnings

warnings.filterwarnings('ignore')
SEED = 42

print("Chargement des données...")
df_train = pd.read_csv('conversion_data_train.csv')
df_test = pd.read_csv('conversion_data_test.csv')

## 1. Définition du Pipeline d'Audit

In [ ]:
class AuditDecisionPipeline(BaseEstimator, ClassifierMixin):
    def __init__(self, base_estimator=None, audit_segments=None, correction_dampening=0.5, audit_cv=5):
        self.base_estimator = base_estimator if base_estimator else xgb.XGBClassifier(random_state=42, use_label_encoder=False, eval_metric='logloss')
        self.audit_segments = audit_segments if audit_segments else ['country', 'source', 'new_user', 'age_bin']
        self.correction_dampening = correction_dampening
        self.audit_cv = audit_cv
        self.corrections_ = {}
        self.classes_ = None
        
    def fit(self, X, y):
        # 1. Train Base Model
        self.base_estimator.fit(X, y)
        if hasattr(self.base_estimator, 'classes_'):
             self.classes_ = self.base_estimator.classes_
        else:
             self.classes_ = np.unique(y)
        
        # 2. OOF Predictions for Audit
        if self.audit_cv > 1:
            try:
                oof_preds = cross_val_predict(clone(self.base_estimator), X, y, cv=self.audit_cv, method='predict_proba')[:, 1]
            except:
                 oof_preds = cross_val_predict(clone(self.base_estimator), X, y, cv=self.audit_cv)
        else:
            oof_preds = self.base_estimator.predict_proba(X)[:, 1]
            
        # 3. Calculate Residuals (Truth - Pred)
        residuals = y - oof_preds
        
        # 4. Audit Segments
        self.corrections_ = {}
        for seg_col in self.audit_segments:
            if seg_col not in X.columns: continue
            
            stats = pd.DataFrame({'feature': X[seg_col], 'residual': residuals}).groupby('feature')['residual'].agg(['mean', 'count', 'std'])
            stats['se'] = stats['std'] / np.sqrt(stats['count'])
            
            # Significant Bias Rule (|Mean| > 1.96 * SE)
            significant = stats[np.abs(stats['mean']) > 1.96 * stats['se']]
            
            if not significant.empty:
                self.corrections_[seg_col] = (significant['mean'] * self.correction_dampening).to_dict()
                
        return self
    
    def predict_proba(self, X):
        base_probs = self.base_estimator.predict_proba(X)[:, 1]
        final_probs = base_probs.copy()
        
        for seg_col, corr_map in self.corrections_.items():
            if seg_col in X.columns:
                adjustments = X[seg_col].map(corr_map).fillna(0).values
                final_probs += adjustments
        
        final_probs = np.clip(final_probs, 0, 1)
        return np.vstack([1-final_probs, final_probs]).T
        
    def predict(self, X):
        return (self.predict_proba(X)[:, 1] >= 0.5).astype(int)

## 2. Préparation et Features d'Audit

In [ ]:
# Preprocessing Standard
for df in [df_train, df_test]:
    df['age_bin'] = pd.cut(df['age'], bins=[0, 18, 25, 30, 35, 40, 45, 50, 60, 100], labels=False).fillna(-1).astype(int)
    df['pages_bin'] = pd.cut(df['total_pages_visited'], bins=[0, 5, 10, 15, 20, 30], labels=False).fillna(-1).astype(int)

cat_cols = ['country', 'source']
for col in cat_cols:
    le = LabelEncoder()
    full_vals = pd.concat([df_train[col], df_test[col]], axis=0)
    le.fit(full_vals)
    df_train[col] = le.transform(df_train[col])
    df_test[col] = le.transform(df_test[col])

features_model = ['country', 'age', 'new_user', 'source', 'total_pages_visited']
features_audit = ['country', 'source', 'new_user', 'age_bin', 'pages_bin']

X = df_train[features_model + ['age_bin', 'pages_bin']]
y = df_train['converted']
X_test = df_test[features_model + ['age_bin', 'pages_bin']]

In [ ]:
print("Training Pipeline...")
pipeline = AuditDecisionPipeline(
    base_estimator=xgb.XGBClassifier(n_estimators=100, learning_rate=0.1, max_depth=6, random_state=42, eval_metric='logloss'),
    audit_segments=features_audit,
    correction_dampening=0.5,
    audit_cv=5
)

pipeline.fit(X, y)

print("Corrections Appliquées :")
for seg, corrections in pipeline.corrections_.items():
    print(f"  Segment '{seg}': {corrections}")